# 02 - Clean and Standardize Urban Renewal Dataset

This notebook reads the raw collected public urban renewal dataset and standardizes it into a lawyer-oriented schema for analysis, scoring, and dashboard use.

It performs field mapping, type cleaning, planning-status normalization, confidence scoring, quality flagging, and report generation.

**Disclaimer:** This notebook organizes public data only. It does not provide legal advice, planning advice, real-estate advice, or binding predictions.

## 1 - Imports and Paths

Import core libraries, define project paths, and create output folders.

In [1]:
from __future__ import annotations

import json
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("default")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

INPUT_PATH = PROCESSED_DIR / "urban_renewal_raw_collected.csv"
OUTPUT_STANDARDIZED_PATH = PROCESSED_DIR / "urban_renewal_standardized.csv"
DATA_DICTIONARY_PATH = METADATA_DIR / "urban_renewal_standardized_data_dictionary.csv"
CLEANING_REPORT_PATH = METADATA_DIR / "cleaning_report.csv"
DATA_QUALITY_ISSUES_PATH = METADATA_DIR / "data_quality_issues.csv"
ANOMALOUS_RECORDS_PATH = METADATA_DIR / "anomalous_records.csv"
CITY_SUMMARY_PATH = PROCESSED_DIR / "urban_renewal_city_summary.csv"
STATUS_SUMMARY_PATH = PROCESSED_DIR / "urban_renewal_status_summary.csv"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, METADATA_DIR, OUTPUTS_DIR, PROJECT_ROOT / "notebooks"]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input path: {INPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Standardized output path: {OUTPUT_STANDARDIZED_PATH.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\Guy\Desktop\Birthday present
Input path: data\processed\urban_renewal_raw_collected.csv
Standardized output path: data\processed\urban_renewal_standardized.csv


## 2 - Load Raw Dataset

Load the raw collected dataset from Notebook 01. The notebook stops clearly if the expected file is missing.

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError("Notebook 01 output not found. Run Notebook 01 first.")

raw_df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns")
print("Raw columns:")
print(list(raw_df.columns))
display(raw_df.head())

Raw shape: 2,988 rows x 38 columns
Raw columns:
['_id', 'MisparMitham', 'Yeshuv', 'SemelYeshuv', 'ShemMitcham', 'YachadKayam', 'YachadTosafti', 'YachadMutza', 'TaarichHachraza', 'MisparTochnit', 'KishurLatar', 'SachHeterim', 'KishurLaMapa', 'Maslul', 'ShnatMatanTokef', 'Bebitzua', 'Status', 'source_name', 'source_url', 'source_type', 'original_resource_id', 'original_file_path', 'original_row_index', 'ingestion_timestamp', 'detected_relevance_reason', 'מפתח לפוליגון תכנית', 'מספר תוכנית', 'שם תוכנית', 'שלב תכנוני', 'יזם תכנון', 'סמל יישוב', 'יישוב', 'תאריך קיום תנאי סף', 'תאריך פרסום להפקדה ברשומות', 'תאריך פרסום לאישור ברשומות', 'קישור לאתר רשות מקרקעי ישראל', 'קישור לאתר מנהל תכנון', 'יחד פוטנציאל לשיווק']


,_id,MisparMitham,Yeshuv,SemelYeshuv,ShemMitcham,YachadKayam,YachadTosafti,YachadMutza,TaarichHachraza,MisparTochnit,...,שלב תכנוני,יזם תכנון,סמל יישוב,יישוב,תאריך קיום תנאי סף,תאריך פרסום להפקדה ברשומות,תאריך פרסום לאישור ברשומות,קישור לאתר רשות מקרקעי ישראל,קישור לאתר מנהל תכנון,יחד פוטנציאל לשיווק
0,1,4001.0,גבעתים ...,6300.0,ערבי נחל ...,126,108,530.0,20/08/2006,גב/490 ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,4005.0,קרית אונו ...,2620.0,ישעיהו ...,198,198,396.0,20/08/2006,תממ/284 ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,4006.0,קרית אונו ...,2620.0,שאול המלך ...,180,48,531.0,27/05/2004,קא/מק/61/285/א ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,4010.0,ראשון לציון ...,8300.0,רמת אליהו (פוזננסקי) ...,58,232,290.0,29/06/2017,413-0292680 ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,4011.0,ראשון לציון ...,8300.0,סלע ...,283,0,1386.0,09/02/2012,רצ/מק/1/13/19/4 ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3 - Helper Functions

Small, explicit helpers for text, URL, numeric, date, yes/no, schema-safe extraction, status normalization, confidence scoring, lawyer notes, and quality flags.

In [3]:
EMPTY_STRINGS = {"", "nan", "none", "null", "-", "--", "n/a", "na"}

TARGET_COLUMNS = [
    "record_id",
    "source_record_id",
    "city",
    "city_code",
    "neighborhood",
    "street_or_area",
    "complex_name",
    "plan_number",
    "renewal_type",
    "planning_status_raw",
    "planning_status_normalized",
    "declared_complex",
    "existing_units",
    "additional_units",
    "proposed_units",
    "permits_total",
    "declaration_date",
    "validity_year",
    "in_execution",
    "mavat_url",
    "map_url",
    "source_name",
    "source_url",
    "source_type",
    "original_resource_id",
    "last_updated",
    "confidence_level",
    "data_confidence_score",
    "data_quality_flag",
    "lawyer_note",
]

TECHNICAL_COLUMNS = [
    "original_file_path",
    "original_row_index",
    "ingestion_timestamp",
    "detected_relevance_reason",
]

FINAL_COLUMNS = TARGET_COLUMNS + TECHNICAL_COLUMNS


def clean_text(value: Any) -> Any:
    """Normalize empty scalar text while preserving Hebrew content."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.lower() in EMPTY_STRINGS:
        return pd.NA
    text = re.sub(r"\s+", " ", text)
    return text if text else pd.NA


def clean_url(value: Any) -> Any:
    """Clean URLs without opening or validating them over the network."""
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).strip()
    if text.lower().startswith(("http://", "https://")):
        return text
    return pd.NA


def parse_numeric(value: Any) -> Any:
    """Parse scalar numeric values; invalid values become pd.NA."""
    cleaned = clean_text(value)
    if pd.isna(cleaned):
        return pd.NA
    text = str(cleaned).replace(",", "")
    numeric = pd.to_numeric(text, errors="coerce")
    if pd.isna(numeric):
        return pd.NA
    numeric_float = float(numeric)
    if numeric_float.is_integer():
        return int(numeric_float)
    return numeric_float


def parse_numeric_series(series: pd.Series) -> pd.Series:
    parsed = series.apply(parse_numeric)
    numeric = pd.to_numeric(parsed, errors="coerce")
    return numeric.astype("Int64")


def parse_date(value: Any) -> Any:
    """Parse Israeli day-first dates and return ISO date strings."""
    cleaned = clean_text(value)
    if pd.isna(cleaned):
        return pd.NA
    parsed = pd.to_datetime(cleaned, dayfirst=True, errors="coerce")
    if pd.isna(parsed):
        return pd.NA
    return parsed.date().isoformat()


def normalize_yes_no(value: Any) -> Any:
    cleaned = clean_text(value)
    if pd.isna(cleaned):
        return pd.NA
    text = str(cleaned).strip().lower()
    truthy = {"כן", "true", "1", "1.0", "yes", "y", "t", "בביצוע"}
    falsy = {"לא", "false", "0", "0.0", "no", "n", "f"}
    if text in truthy:
        return True
    if text in falsy:
        return False
    return pd.NA


def column_exists(df: pd.DataFrame, column: str) -> bool:
    return column in df.columns


def get_series_or_na(df: pd.DataFrame, column: str) -> pd.Series:
    if column_exists(df, column):
        return df[column]
    return pd.Series(pd.NA, index=df.index, dtype="object")


def first_non_missing(*values: Any) -> Any:
    for value in values:
        if not pd.isna(value):
            return value
    return pd.NA


def normalize_identifier(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text)
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text


def has_value(value: Any) -> bool:
    return not pd.isna(value)


def source_is_official_public(row: pd.Series) -> bool:
    source_blob = " ".join(
        str(row.get(col, "")) for col in ["source_name", "source_url", "source_type", "original_resource_id"]
    ).lower()
    official_terms = ["data.gov", "gov", "government", "official", "ממשל", "משרד", "רשות"]
    return any(term in source_blob for term in official_terms)


def normalize_planning_status(row: pd.Series) -> pd.Series:
    raw = clean_text(row.get("planning_status_raw"))
    in_execution = row.get("in_execution")
    status = "UNKNOWN"
    reason = "planning_status_raw is missing or unclear"

    if not pd.isna(raw):
        text = str(raw)
        if "תכנון ראשוני" in text:
            status = "PLAN_IN_PROGRESS"
            reason = "planning_status_raw contains תכנון ראשוני"
        elif "תכנון סטטוטורי" in text:
            status = "PLAN_IN_PROGRESS"
            reason = "planning_status_raw contains תכנון סטטוטורי"
        elif "מאושרת לפני מימוש" in text:
            status = "PLAN_APPROVED"
            reason = "planning_status_raw contains מאושרת לפני מימוש"
        elif "אחרי רישוי" in text:
            status = "PERMIT_APPROVED"
            reason = "planning_status_raw contains אחרי רישוי"
        elif "במימוש" in text:
            status = "CONSTRUCTION"
            reason = "planning_status_raw contains במימוש"
        else:
            status = "UNKNOWN"
            reason = "planning_status_raw did not match conservative normalization rules"

    if in_execution is True and status in {"UNKNOWN", "PLAN_APPROVED", "PERMIT_REQUESTED", "PERMIT_APPROVED"}:
        previous = status
        status = "CONSTRUCTION"
        reason = f"in_execution is True; overridden from {previous}"

    return pd.Series({"planning_status_normalized": status, "status_normalization_reason": reason})


def create_lawyer_note(row: pd.Series) -> str:
    status = row.get("planning_status_normalized", "UNKNOWN")
    notes_by_status = {
        "PLAN_APPROVED": "Official urban renewal record found. Approved planning status appears in the public source; recommended to inspect Mavat documents.",
        "PERMIT_APPROVED": "Official record found with indication of post-licensing/permit stage; worth deeper legal and planning review.",
        "CONSTRUCTION": "Official record indicates implementation/construction stage; verify current status in Mavat and municipal systems.",
        "PLAN_IN_PROGRESS": "Official renewal record found. Planning appears to be in progress; check plan documents and latest public status.",
        "DECLARED_COMPLEX": "Declared urban renewal complex found; verify current planning stage and related plan documents.",
        "UNKNOWN": "Official record found, but planning status is unclear or missing; manual verification required.",
    }
    note = notes_by_status.get(status, notes_by_status["UNKNOWN"])
    if has_value(row.get("plan_number")):
        note += " Plan number exists; inspect Mavat documents."
    if has_value(row.get("mavat_url")):
        note += " Mavat link is available."
    return note


def calculate_data_confidence_score(row: pd.Series) -> int:
    score = 0
    if source_is_official_public(row):
        score += 25
    if has_value(row.get("city")):
        score += 15
    if has_value(row.get("complex_name")):
        score += 15
    if has_value(row.get("planning_status_raw")):
        score += 15
    if has_value(row.get("plan_number")):
        score += 10
    if has_value(row.get("mavat_url")):
        score += 10
    if has_value(row.get("map_url")):
        score += 5
    if any(has_value(row.get(col)) for col in ["existing_units", "additional_units", "proposed_units"]):
        score += 5
    if row.get("data_quality_flag") != "OK":
        score -= 20
    if row.get("planning_status_normalized") == "UNKNOWN":
        score -= 10
    if not has_value(row.get("source_url")):
        score -= 10
    return int(max(0, min(100, score)))


def assign_confidence_level(score: Any) -> str:
    if pd.isna(score):
        return "LOW"
    score = int(score)
    if score >= 80:
        return "HIGH"
    if score >= 50:
        return "MEDIUM"
    return "LOW"


def build_data_quality_flags(row: pd.Series) -> str:
    flags: List[str] = []
    if not has_value(row.get("city")):
        flags.append("MISSING_CITY")
    if not has_value(row.get("complex_name")):
        flags.append("MISSING_COMPLEX_NAME")
    if not has_value(row.get("planning_status_raw")):
        flags.append("MISSING_STATUS")
    if not has_value(row.get("mavat_url")) and not has_value(row.get("map_url")):
        flags.append("MISSING_LINKS")
    if bool(row.get("_invalid_numeric_units", False)):
        flags.append("INVALID_NUMERIC_UNITS")

    unit_fields_for_negative_check = ["existing_units", "additional_units", "proposed_units", "permits_total"]
    if any(has_value(row.get(col)) and row.get(col) < 0 for col in unit_fields_for_negative_check):
        flags.append("NEGATIVE_UNIT_VALUE")

    existing_units = row.get("existing_units")
    proposed_units = row.get("proposed_units")
    if has_value(existing_units) and has_value(proposed_units) and proposed_units < existing_units:
        flags.append("UNIT_LOGIC_ANOMALY")

    source_record_id = str(row.get("source_record_id") or "")
    complex_name = str(row.get("complex_name") or "")
    if source_record_id == "5009236" or "קריניצי 109" in complex_name:
        flags.append("COLUMN_SHIFT_OR_SOURCE_ANOMALY")

    if bool(row.get("_duplicate_record_id", False)):
        flags.append("DUPLICATE_RECORD_ID")

    return "|".join(flags) if flags else "OK"

## 4 - Standardize Columns

Map the raw data.gov.il columns into the target schema. Missing source columns are filled with pd.NA rather than invented values.

In [4]:
standardized_df = pd.DataFrame(index=raw_df.index)

COLUMN_MAPPING = {
    "MisparMitham": "source_record_id",
    "Yeshuv": "city",
    "SemelYeshuv": "city_code",
    "ShemMitcham": "complex_name",
    "YachadKayam": "existing_units",
    "YachadTosafti": "additional_units",
    "YachadMutza": "proposed_units",
    "TaarichHachraza": "declaration_date",
    "MisparTochnit": "plan_number",
    "KishurLatar": "mavat_url",
    "SachHeterim": "permits_total",
    "KishurLaMapa": "map_url",
    "Maslul": "renewal_type",
    "ShnatMatanTokef": "validity_year",
    "Bebitzua": "in_execution",
    "Status": "planning_status_raw",
}

for source_col, target_col in COLUMN_MAPPING.items():
    standardized_df[target_col] = get_series_or_na(raw_df, source_col)

for col in ["neighborhood", "street_or_area"]:
    standardized_df[col] = pd.NA

for col in ["source_name", "source_type", "original_resource_id", "original_file_path", "original_row_index", "ingestion_timestamp", "detected_relevance_reason"]:
    standardized_df[col] = get_series_or_na(raw_df, col)

raw_source_url = get_series_or_na(raw_df, "source_url")

raw_numeric_source_values = {
    col: standardized_df[col].copy()
    for col in ["existing_units", "additional_units", "proposed_units", "permits_total", "validity_year", "city_code"]
}

print(f"Standardized working shape: {standardized_df.shape}")
display(standardized_df.head())

Standardized working shape: (2988, 25)


,source_record_id,city,city_code,complex_name,existing_units,additional_units,proposed_units,declaration_date,plan_number,mavat_url,...,planning_status_raw,neighborhood,street_or_area,source_name,source_type,original_resource_id,original_file_path,original_row_index,ingestion_timestamp,detected_relevance_reason
0,4001.0,גבעתים ...,6300.0,ערבי נחל ...,126,108,530.0,20/08/2006,גב/490 ...,https://mavat.iplan.gov.il/SV4/1/5073314/310 ...,...,תכנית מאושרת - אחרי רישוי ...,<NA>,<NA>,data.gov.il - מתחמי התחדשות עירונית,government_open_data_resource,f65a0daf-f737-49c5-9424-d378d52104f5,data\raw\data_gov_f65a0daf-f737-49c5-9424-d378...,0,2026-05-27T16:09:42,Known official data.gov.il urban renewal resource
1,4005.0,קרית אונו ...,2620.0,ישעיהו ...,198,198,396.0,20/08/2006,תממ/284 ...,https://mavat.iplan.gov.il/SV4/1/5048159/310 ...,...,תכנית מאושרת במימוש ...,<NA>,<NA>,data.gov.il - מתחמי התחדשות עירונית,government_open_data_resource,f65a0daf-f737-49c5-9424-d378d52104f5,data\raw\data_gov_f65a0daf-f737-49c5-9424-d378...,1,2026-05-27T16:09:42,Known official data.gov.il urban renewal resource
2,4006.0,קרית אונו ...,2620.0,שאול המלך ...,180,48,531.0,27/05/2004,קא/מק/61/285/א ...,https://mavat.iplan.gov.il/SV4/1/5052333/310 ...,...,תכנית מאושרת - אחרי רישוי ...,<NA>,<NA>,data.gov.il - מתחמי התחדשות עירונית,government_open_data_resource,f65a0daf-f737-49c5-9424-d378d52104f5,data\raw\data_gov_f65a0daf-f737-49c5-9424-d378...,2,2026-05-27T16:09:42,Known official data.gov.il urban renewal resource
3,4010.0,ראשון לציון ...,8300.0,רמת אליהו (פוזננסקי) ...,58,232,290.0,29/06/2017,413-0292680 ...,https://mavat.iplan.gov.il/SV4/1/4000346997/31...,...,תכנית מאושרת במימוש ...,<NA>,<NA>,data.gov.il - מתחמי התחדשות עירונית,government_open_data_resource,f65a0daf-f737-49c5-9424-d378d52104f5,data\raw\data_gov_f65a0daf-f737-49c5-9424-d378...,3,2026-05-27T16:09:42,Known official data.gov.il urban renewal resource
4,4011.0,ראשון לציון ...,8300.0,סלע ...,283,0,1386.0,09/02/2012,רצ/מק/1/13/19/4 ...,https://mavat.iplan.gov.il/SV4/1/4097824/310 ...,...,תכנית מאושרת במימוש ...,<NA>,<NA>,data.gov.il - מתחמי התחדשות עירונית,government_open_data_resource,f65a0daf-f737-49c5-9424-d378d52104f5,data\raw\data_gov_f65a0daf-f737-49c5-9424-d378...,4,2026-05-27T16:09:42,Known official data.gov.il urban renewal resource


## 5 - Clean Types and Values

Clean text, URLs, numeric fields, dates, and yes/no indicators. Invalid numeric values become missing values and are later flagged.

In [5]:
text_columns = [
    "source_record_id",
    "city",
    "neighborhood",
    "street_or_area",
    "complex_name",
    "plan_number",
    "renewal_type",
    "planning_status_raw",
    "source_name",
    "source_type",
    "original_resource_id",
    "original_file_path",
    "detected_relevance_reason",
]

for col in text_columns:
    standardized_df[col] = standardized_df[col].apply(clean_text)

standardized_df["source_record_id"] = standardized_df["source_record_id"].apply(normalize_identifier)
standardized_df["plan_number"] = standardized_df["plan_number"].apply(normalize_identifier)
standardized_df["original_resource_id"] = standardized_df["original_resource_id"].apply(normalize_identifier)

url_columns = ["mavat_url", "map_url"]
for col in url_columns:
    standardized_df[col] = standardized_df[col].apply(clean_url)

raw_source_url_clean = raw_source_url.apply(clean_url)
standardized_df["source_url"] = [
    first_non_missing(mavat, source, map_url)
    for mavat, source, map_url in zip(standardized_df["mavat_url"], raw_source_url_clean, standardized_df["map_url"])
]

numeric_columns = ["existing_units", "additional_units", "proposed_units", "permits_total", "validity_year", "city_code"]
invalid_numeric_masks: Dict[str, pd.Series] = {}
for col in numeric_columns:
    raw_clean = raw_numeric_source_values[col].apply(clean_text)
    parsed = parse_numeric_series(raw_numeric_source_values[col])
    invalid_numeric_masks[col] = raw_clean.notna() & parsed.isna()
    standardized_df[col] = parsed

unit_invalid_columns = ["existing_units", "additional_units", "proposed_units"]
standardized_df["_invalid_numeric_units"] = pd.concat(
    [invalid_numeric_masks[col] for col in unit_invalid_columns], axis=1
).any(axis=1)

standardized_df["declaration_date"] = standardized_df["declaration_date"].apply(parse_date)
standardized_df["ingestion_timestamp"] = standardized_df["ingestion_timestamp"].apply(clean_text)
standardized_df["last_updated"] = [
    first_non_missing(parse_date(ingested), declaration)
    for ingested, declaration in zip(standardized_df["ingestion_timestamp"], standardized_df["declaration_date"])
]

standardized_df["in_execution"] = standardized_df["in_execution"].apply(normalize_yes_no)
standardized_df["original_row_index"] = parse_numeric_series(standardized_df["original_row_index"])

standardized_df["record_id"] = [
    f"UR_{source_id}" if has_value(source_id) else f"UR_ROW_{idx}"
    for idx, source_id in zip(standardized_df.index, standardized_df["source_record_id"])
]

print("Cleaned value preview:")
display(standardized_df[["record_id", "city", "complex_name", "planning_status_raw", "existing_units", "proposed_units"]].head())

C:\Users\Guy\AppData\Local\Temp\ipykernel_32236\828491402.py:94: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(cleaned, dayfirst=True, errors="coerce")


Cleaned value preview:


,record_id,city,complex_name,planning_status_raw,existing_units,proposed_units
0,UR_4001,גבעתים,ערבי נחל,תכנית מאושרת - אחרי רישוי,126,530
1,UR_4005,קרית אונו,ישעיהו,תכנית מאושרת במימוש,198,396
2,UR_4006,קרית אונו,שאול המלך,תכנית מאושרת - אחרי רישוי,180,531
3,UR_4010,ראשון לציון,רמת אליהו (פוזננסקי),תכנית מאושרת במימוש,58,290
4,UR_4011,ראשון לציון,סלע,תכנית מאושרת במימוש,283,1386


## 6 - Normalize Planning Status

Convert conservative Hebrew planning-status phrases into normalized planning categories without overclaiming uncertain stages.

In [6]:
status_normalized = standardized_df.apply(normalize_planning_status, axis=1)
standardized_df["planning_status_normalized"] = status_normalized["planning_status_normalized"]
standardized_df["status_normalization_reason"] = status_normalized["status_normalization_reason"]

print("Raw status counts:")
display(standardized_df["planning_status_raw"].value_counts(dropna=False).rename_axis("planning_status_raw").reset_index(name="num_records"))

print("Normalized status counts:")
display(standardized_df["planning_status_normalized"].value_counts(dropna=False).rename_axis("planning_status_normalized").reset_index(name="num_records"))

Raw status counts:


,planning_status_raw,num_records
0,<NA>,1112
1,תכנית מאושרת לפני מימוש,650
2,תכנון סטטוטורי,530
3,תכנית מאושרת - אחרי רישוי,346
4,תכנון ראשוני,240
5,תכנית מאושרת במימוש,108
6,2025,2


Normalized status counts:


,planning_status_normalized,num_records
0,UNKNOWN,1114
1,PLAN_IN_PROGRESS,770
2,PLAN_APPROVED,650
3,PERMIT_APPROVED,346
4,CONSTRUCTION,108


## 7 - Declared Complex, Confidence, and Lawyer Notes

Create declared-complex indicators and short practical notes. Final confidence scores are calculated after quality flags are available.

In [7]:
def infer_declared_complex(row: pd.Series) -> Any:
    source_name = str(row.get("source_name") or "").lower()
    if "מתחמי התחדשות עירונית" in source_name or "urban renewal" in source_name:
        return True
    if has_value(row.get("source_record_id")):
        return True
    if has_value(row.get("declaration_date")):
        return True
    if has_value(row.get("validity_year")):
        return True
    return pd.NA

standardized_df["declared_complex"] = standardized_df.apply(infer_declared_complex, axis=1)
standardized_df["lawyer_note"] = standardized_df.apply(create_lawyer_note, axis=1)

print("Declared complex counts:")
display(standardized_df["declared_complex"].value_counts(dropna=False).rename_axis("declared_complex").reset_index(name="num_records"))
display(standardized_df[["record_id", "planning_status_normalized", "declared_complex", "lawyer_note"]].head())

Declared complex counts:


,declared_complex,num_records
0,True,1876
1,<NA>,1112


,record_id,planning_status_normalized,declared_complex,lawyer_note
0,UR_4001,PERMIT_APPROVED,True,Official record found with indication of post-...
1,UR_4005,CONSTRUCTION,True,Official record indicates implementation/const...
2,UR_4006,PERMIT_APPROVED,True,Official record found with indication of post-...
3,UR_4010,CONSTRUCTION,True,Official record indicates implementation/const...
4,UR_4011,CONSTRUCTION,True,Official record indicates implementation/const...


## 8 - Data Quality Checks

Flag missing core fields, invalid numeric fields, unit logic anomalies, known source anomalies, and duplicate record IDs. Records are kept by default and reported separately.

In [8]:
standardized_df["_duplicate_record_id"] = standardized_df["record_id"].duplicated(keep=False)
standardized_df["data_quality_flag"] = standardized_df.apply(build_data_quality_flags, axis=1)
standardized_df["has_quality_issue"] = standardized_df["data_quality_flag"] != "OK"

standardized_df["data_confidence_score"] = standardized_df.apply(calculate_data_confidence_score, axis=1)
standardized_df["confidence_level"] = standardized_df["data_confidence_score"].apply(assign_confidence_level)

quality_issue_columns = [
    "record_id",
    "source_record_id",
    "city",
    "complex_name",
    "plan_number",
    "planning_status_raw",
    "planning_status_normalized",
    "data_quality_flag",
    "confidence_level",
    "lawyer_note",
]

data_quality_issues_df = standardized_df.loc[standardized_df["has_quality_issue"], quality_issue_columns].copy()
data_quality_issues_df.to_csv(DATA_QUALITY_ISSUES_PATH, index=False, encoding="utf-8-sig")

anomaly_terms = ["COLUMN_SHIFT_OR_SOURCE_ANOMALY", "UNIT_LOGIC_ANOMALY", "DUPLICATE_RECORD_ID"]
anomalous_records_df = standardized_df[
    standardized_df["data_quality_flag"].apply(lambda text: any(term in str(text) for term in anomaly_terms))
].copy()
anomalous_records_df.to_csv(ANOMALOUS_RECORDS_PATH, index=False, encoding="utf-8-sig")

print(f"Records with quality issues: {len(data_quality_issues_df):,}")
print(f"Anomalous records: {len(anomalous_records_df):,}")
display(standardized_df["data_quality_flag"].value_counts(dropna=False).rename_axis("data_quality_flag").reset_index(name="num_records").head(20))

TypeError: boolean value of NA is ambiguous

## 9 - Final Schema Ordering and Save

Order the standardized columns exactly as requested and save the main processed CSV.

In [ ]:
from pathlib import Path
import time

for col in FINAL_COLUMNS:
    if col not in standardized_df.columns:
        standardized_df[col] = pd.NA

standardized_final_df = standardized_df[FINAL_COLUMNS].copy()

# Make sure output folder exists
OUTPUT_STANDARDIZED_PATH.parent.mkdir(parents=True, exist_ok=True)

try:
    standardized_final_df.to_csv(
        OUTPUT_STANDARDIZED_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved standardized dataset: {OUTPUT_STANDARDIZED_PATH.relative_to(PROJECT_ROOT)}")

except PermissionError:
    # Windows often locks CSV files if they are open in Excel / VS Code preview.
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    fallback_path = OUTPUT_STANDARDIZED_PATH.with_name(
        f"{OUTPUT_STANDARDIZED_PATH.stem}_{timestamp}{OUTPUT_STANDARDIZED_PATH.suffix}"
    )

    standardized_final_df.to_csv(
        fallback_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("PermissionError: the original output file is probably open or locked.")
    print(f"Saved instead to fallback file: {fallback_path.relative_to(PROJECT_ROOT)}")
    print("Close the locked CSV file and rerun this cell later if you want to overwrite the original path.")

print(f"Standardized shape: {standardized_final_df.shape[0]:,} rows x {standardized_final_df.shape[1]:,} columns")
display(standardized_final_df.head())

Saved standardized dataset: data\processed\urban_renewal_standardized.csv
Standardized shape: 938 rows x 34 columns


,record_id,source_record_id,city,city_code,neighborhood,street_or_area,complex_name,plan_number,renewal_type,planning_status_raw,...,original_resource_id,last_updated,confidence_level,data_confidence_score,data_quality_flag,lawyer_note,original_file_path,original_row_index,ingestion_timestamp,detected_relevance_reason
0,UR_4001,4001,גבעתים,6300,<NA>,<NA>,ערבי נחל,גב/490,מיסוי,תכנית מאושרת - אחרי רישוי,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record found with indication of post-...,data\manual_sources\data_gov_f65a0daf_records....,0,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
1,UR_4005,4005,קרית אונו,2620,<NA>,<NA>,ישעיהו,תממ/284,מיסוי,תכנית מאושרת במימוש,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record indicates implementation/const...,data\manual_sources\data_gov_f65a0daf_records....,1,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
2,UR_4006,4006,קרית אונו,2620,<NA>,<NA>,שאול המלך,קא/מק/61/285/א,מיסוי,תכנית מאושרת - אחרי רישוי,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record found with indication of post-...,data\manual_sources\data_gov_f65a0daf_records....,2,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
3,UR_4010,4010,ראשון לציון,8300,<NA>,<NA>,רמת אליהו (פוזננסקי),413-0292680,מיסוי,תכנית מאושרת במימוש,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record indicates implementation/const...,data\manual_sources\data_gov_f65a0daf_records....,3,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
4,UR_4011,4011,ראשון לציון,8300,<NA>,<NA>,סלע,רצ/מק/1/13/19/4,מיסוי,תכנית מאושרת במימוש,...,f65a0daf-f737-49c5-9424-d378d52104f5,2026-05-26,HIGH,100,OK,Official record indicates implementation/const...,data\manual_sources\data_gov_f65a0daf_records....,4,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...


## 10 - Reports and Summaries

Create the data dictionary, cleaning report, city summary, and status summary for downstream notebooks and dashboard work.

In [ ]:
DATA_DICTIONARY_ENTRIES = [
    ("record_id", "Stable local record identifier for this project.", "derived from MisparMitham or row index", "string", "UR_5000000", "Not a legal identifier."),
    ("source_record_id", "Original source record or complex identifier.", "MisparMitham", "string", None, "Preserved from public source when available."),
    ("city", "City or locality name.", "Yeshuv", "string", None, "Hebrew text preserved."),
    ("city_code", "Official locality code when supplied.", "SemelYeshuv", "nullable integer", None, "No values invented."),
    ("neighborhood", "Neighborhood name if later available.", None, "string", None, "Currently unavailable in source."),
    ("street_or_area", "Street, address, or area label if later available.", None, "string", None, "Currently unavailable in source."),
    ("complex_name", "Urban renewal complex name.", "ShemMitcham", "string", None, "Core lawyer-facing label."),
    ("plan_number", "Planning file or plan number.", "MisparTochnit", "string", None, "Inspect in Mavat when present."),
    ("renewal_type", "Raw renewal route/type.", "Maslul", "string", None, "Not translated or legally interpreted."),
    ("planning_status_raw", "Raw planning status from source.", "Status", "string", None, "Hebrew value preserved."),
    ("planning_status_normalized", "Conservative normalized planning status.", "Status and Bebitzua", "category", "PLAN_APPROVED", "Uses explicit mapping only."),
    ("declared_complex", "Whether record appears to be a declared urban renewal complex.", "derived", "boolean", "True", "Robust heuristic, not legal advice."),
    ("existing_units", "Existing housing units.", "YachadKayam", "nullable integer", None, "Invalid text becomes missing and is flagged."),
    ("additional_units", "Additional housing units.", "YachadTosafti", "nullable integer", None, "Invalid text becomes missing and is flagged."),
    ("proposed_units", "Proposed housing units.", "YachadMutza", "nullable integer", None, "Invalid text becomes missing and is flagged."),
    ("permits_total", "Total permits count when supplied.", "SachHeterim", "nullable integer", None, "No permit interpretation is added."),
    ("declaration_date", "Declaration date as ISO date.", "TaarichHachraza", "date string", "2025-04-28", "Parsed day-first."),
    ("validity_year", "Year of plan validity or effect when supplied.", "ShnatMatanTokef", "nullable integer", None, "Source meaning should be verified."),
    ("in_execution", "Whether source indicates execution/implementation.", "Bebitzua", "boolean", "True", "Normalized yes/no."),
    ("mavat_url", "Mavat or planning source URL.", "KishurLatar", "URL string", None, "Not opened by this notebook."),
    ("map_url", "Map URL when supplied.", "KishurLaMapa", "URL string", None, "Not opened by this notebook."),
    ("source_name", "Source dataset name.", "source_name", "string", None, "From Notebook 01 metadata."),
    ("source_url", "Preferred source URL for review.", "KishurLatar/source_url/KishurLaMapa", "URL string", None, "Prefers Mavat URL."),
    ("source_type", "Source type metadata.", "source_type", "string", None, "From Notebook 01."),
    ("original_resource_id", "Original CKAN resource ID.", "original_resource_id", "string", None, "From Notebook 01."),
    ("last_updated", "Best available last update date.", "ingestion_timestamp or TaarichHachraza", "date string", None, "Uses ingestion date if available."),
    ("confidence_level", "HIGH/MEDIUM/LOW confidence label.", "derived", "category", "HIGH", "Operational data confidence, not legal confidence."),
    ("data_confidence_score", "0-100 data confidence score.", "derived", "integer", "85", "Based on completeness and quality flags."),
    ("data_quality_flag", "Pipe-separated data quality flags.", "derived", "string", "OK", "Records are retained by default."),
    ("lawyer_note", "Short cautious review note.", "derived", "string", None, "Does not provide legal advice."),
    ("original_file_path", "Raw file path from Notebook 01.", "original_file_path", "string", None, "Technical provenance."),
    ("original_row_index", "Original row index in source data.", "original_row_index", "nullable integer", None, "Technical provenance."),
    ("ingestion_timestamp", "Notebook 01 ingestion timestamp.", "ingestion_timestamp", "string", None, "Technical provenance."),
    ("detected_relevance_reason", "Reason Notebook 01 considered the source relevant.", "detected_relevance_reason", "string", None, "Technical provenance."),
]

data_dictionary_df = pd.DataFrame(
    DATA_DICTIONARY_ENTRIES,
    columns=["column_name", "description", "source_column", "data_type", "example", "notes"],
)

examples = {}
for col in data_dictionary_df["column_name"]:
    if col in standardized_final_df.columns:
        non_missing = standardized_final_df[col].dropna()
        if not non_missing.empty:
            examples[col] = str(non_missing.iloc[0])

data_dictionary_df["example"] = data_dictionary_df.apply(
    lambda row: examples.get(row["column_name"], row["example"]), axis=1
)
data_dictionary_df.to_csv(DATA_DICTIONARY_PATH, index=False, encoding="utf-8-sig")

cleaning_metrics = [
    ("input_rows", len(raw_df), "Rows read from Notebook 01 raw collected CSV."),
    ("input_columns", raw_df.shape[1], "Columns read from Notebook 01 raw collected CSV."),
    ("output_rows", len(standardized_final_df), "Rows written to standardized dataset."),
    ("output_columns", standardized_final_df.shape[1], "Columns written to standardized dataset."),
    ("duplicate_record_ids", int(standardized_df["_duplicate_record_id"].sum()), "Rows whose standardized record_id is duplicated."),
    ("records_with_quality_issues", int(standardized_df["has_quality_issue"].sum()), "Rows where data_quality_flag is not OK."),
    ("records_missing_city", int(standardized_df["city"].isna().sum()), "Rows missing city."),
    ("records_missing_complex_name", int(standardized_df["complex_name"].isna().sum()), "Rows missing complex name."),
    ("records_missing_status", int(standardized_df["planning_status_raw"].isna().sum()), "Rows missing raw planning status."),
    ("records_with_plan_number", int(standardized_df["plan_number"].notna().sum()), "Rows with plan number."),
    ("records_with_mavat_url", int(standardized_df["mavat_url"].notna().sum()), "Rows with Mavat/source URL."),
    ("records_with_map_url", int(standardized_df["map_url"].notna().sum()), "Rows with map URL."),
    ("records_with_existing_units", int(standardized_df["existing_units"].notna().sum()), "Rows with existing units."),
    ("records_with_proposed_units", int(standardized_df["proposed_units"].notna().sum()), "Rows with proposed units."),
    ("number_of_cities", int(standardized_df["city"].nunique(dropna=True)), "Distinct cities."),
    ("number_of_status_categories_raw", int(standardized_df["planning_status_raw"].nunique(dropna=True)), "Distinct raw statuses."),
    ("number_of_status_categories_normalized", int(standardized_df["planning_status_normalized"].nunique(dropna=True)), "Distinct normalized statuses."),
    ("high_confidence_records", int((standardized_df["confidence_level"] == "HIGH").sum()), "Rows with HIGH confidence."),
    ("medium_confidence_records", int((standardized_df["confidence_level"] == "MEDIUM").sum()), "Rows with MEDIUM confidence."),
    ("low_confidence_records", int((standardized_df["confidence_level"] == "LOW").sum()), "Rows with LOW confidence."),
]

cleaning_report_df = pd.DataFrame(cleaning_metrics, columns=["metric", "value", "notes"])
cleaning_report_df.to_csv(CLEANING_REPORT_PATH, index=False, encoding="utf-8-sig")

city_group = standardized_final_df.groupby("city", dropna=False)
city_summary_df = city_group.agg(
    num_records=("record_id", "size"),
    num_with_plan_number=("plan_number", lambda s: int(s.notna().sum())),
    num_high_confidence=("confidence_level", lambda s: int((s == "HIGH").sum())),
    total_existing_units=("existing_units", "sum"),
    total_proposed_units=("proposed_units", "sum"),
).reset_index()
city_summary_df["num_plan_approved"] = city_group["planning_status_normalized"].apply(lambda s: int((s == "PLAN_APPROVED").sum())).values
city_summary_df["num_permit_approved"] = city_group["planning_status_normalized"].apply(lambda s: int((s == "PERMIT_APPROVED").sum())).values
city_summary_df["num_construction"] = city_group["planning_status_normalized"].apply(lambda s: int((s == "CONSTRUCTION").sum())).values
city_summary_df = city_summary_df[
    [
        "city",
        "num_records",
        "num_plan_approved",
        "num_permit_approved",
        "num_construction",
        "num_with_plan_number",
        "num_high_confidence",
        "total_existing_units",
        "total_proposed_units",
    ]
]
city_summary_df.to_csv(CITY_SUMMARY_PATH, index=False, encoding="utf-8-sig")

status_summary_df = (
    standardized_final_df.groupby("planning_status_normalized", dropna=False)
    .agg(
        num_records=("record_id", "size"),
        avg_data_confidence_score=("data_confidence_score", "mean"),
    )
    .reset_index()
)
status_summary_df["share_of_records"] = status_summary_df["num_records"] / len(standardized_final_df) if len(standardized_final_df) else 0
status_summary_df = status_summary_df[
    ["planning_status_normalized", "num_records", "share_of_records", "avg_data_confidence_score"]
]
status_summary_df.to_csv(STATUS_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("Reports saved:")
for path in [DATA_DICTIONARY_PATH, CLEANING_REPORT_PATH, CITY_SUMMARY_PATH, STATUS_SUMMARY_PATH, DATA_QUALITY_ISSUES_PATH, ANOMALOUS_RECORDS_PATH]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

Reports saved:
- data\metadata\urban_renewal_standardized_data_dictionary.csv
- data\metadata\cleaning_report.csv
- data\processed\urban_renewal_city_summary.csv
- data\processed\urban_renewal_status_summary.csv
- data\metadata\data_quality_issues.csv
- data\metadata\anomalous_records.csv


## 11 - Final Summary

Summarize the standardized dataset and confirm whether downstream scoring and dashboard work can begin.

In [ ]:
summary = {
    "input_rows": len(raw_df),
    "output_rows": len(standardized_final_df),
    "number_of_cities": int(standardized_final_df["city"].nunique(dropna=True)),
    "raw_statuses": int(standardized_df["planning_status_raw"].nunique(dropna=True)),
    "normalized_statuses": int(standardized_final_df["planning_status_normalized"].nunique(dropna=True)),
    "quality_issue_count": int((standardized_df["data_quality_flag"] != "OK").sum()),
}

print("Notebook 02 final summary")
print("=" * 80)
for key, value in summary.items():
    print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")

print("\nOutput paths:")
for path in [
    OUTPUT_STANDARDIZED_PATH,
    DATA_DICTIONARY_PATH,
    CLEANING_REPORT_PATH,
    DATA_QUALITY_ISSUES_PATH,
    ANOMALOUS_RECORDS_PATH,
    CITY_SUMMARY_PATH,
    STATUS_SUMMARY_PATH,
]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

if OUTPUT_STANDARDIZED_PATH.exists() and len(standardized_final_df) == len(raw_df):
    print("\nNotebook 03 can start: standardized dataset exists and row count matches the raw input.")
else:
    print("\nNotebook 03 should wait: verify standardized output and row counts first.")

print("Notebook 02 completed. The standardized dataset is ready for Notebook 03 scoring and the Streamlit dashboard.")

Notebook 02 final summary
input_rows: 938
output_rows: 938
number_of_cities: 76
raw_statuses: 6
normalized_statuses: 5
quality_issue_count: 2

Output paths:
- data\processed\urban_renewal_standardized.csv
- data\metadata\urban_renewal_standardized_data_dictionary.csv
- data\metadata\cleaning_report.csv
- data\metadata\data_quality_issues.csv
- data\metadata\anomalous_records.csv
- data\processed\urban_renewal_city_summary.csv
- data\processed\urban_renewal_status_summary.csv

Notebook 03 can start: standardized dataset exists and row count matches the raw input.
Notebook 02 completed. The standardized dataset is ready for Notebook 03 scoring and the Streamlit dashboard.
